# 01 — Data audit: what is in the data, and can we trust the split?

**Work package:** WP1 · **Runtime: CPU is fine** (about 5 minutes) — no GPU quota used.

Questions (PROJECT_PLAN §2, DECISION_LOG D-008):
1. **Coverage** — how big are the defects, where are they, which classes co-occur?
2. **Augmentation accounting** — how many real photographs sit behind the 3,349 training images?
3. **Direct leakage** — does any original photo appear in more than one split?
4. **Near-duplicate leakage** — are VALID/TEST images near-identical to training images
   (e.g. neighbouring video frames)? If so, test scores would be optimistic.

Learning objectives: data selection, leakage, random vs grouped splits (LO 16).

## 1. Setup

In [ ]:
import os, sys, json, subprocess
REPO_URL = "https://github.com/Hudson11111/MMA3001Project.git"
REPO_DIR = "/content/MMA3001Project"
if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} fetch -q origin && git -C {REPO_DIR} checkout -q main && git -C {REPO_DIR} reset -q --hard origin/main
else:
    !git clone -q {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pip install -q -r requirements-colab.txt
if f"{REPO_DIR}/src" not in sys.path:
    sys.path.insert(0, f"{REPO_DIR}/src")

from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from porkqc import CLASS_NAMES
from porkqc.config import load_yaml, check_class_names
from porkqc.data.download import download_dataset, fix_data_yaml, read_class_names

cfg = load_yaml("configs/dataset.yaml")
rf = cfg["roboflow"]
data_root = Path(cfg["paths"]["colab_data_dir"])
if not (data_root / "data.yaml").exists():
    data_root = download_dataset(rf["workspace"], rf["project"], rf["version"], data_root, export_format=rf["export_format"])
fix_data_yaml(data_root / "data.yaml")
check_class_names(read_class_names(data_root / "data.yaml"), CLASS_NAMES)
SPLITS = ("train", "valid", "test")
FIG = Path("results/figures"); FIG.mkdir(parents=True, exist_ok=True)
AUD = Path("results/audit"); AUD.mkdir(parents=True, exist_ok=True)
git_commit = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
print("data:", data_root, "| commit:", git_commit)

## 2. Split overview (recap of notebook 00)

In [ ]:
from porkqc.data.audit import summarise_dataset
summ = summarise_dataset(data_root, len(CLASS_NAMES))
overview = pd.DataFrame([{ "split": k, "images": s.n_images, "unique originals": s.n_unique_originals,
    "null images": s.n_null_images, "objects": s.n_instances,
    "missing/invalid labels": s.n_missing_labels + s.n_invalid_label_files} for k, s in summ.items()]).set_index("split")
per_class = pd.DataFrame({k: s.instances_per_class for k, s in summ.items()}, index=CLASS_NAMES)
display(overview); display(per_class)

## 3. Box statistics per class

Sizes are fractions of the image area. "Small" uses the COCO definition (< 32 × 32 px,
i.e. < 0.26% of a 720 × 540 image) — small objects are typically hardest to detect.

In [ ]:
from porkqc.data.audit import box_table
boxes = pd.DataFrame([r for s in SPLITS for r in box_table(data_root / s, len(CLASS_NAMES), split=s)])
boxes["class"] = boxes["class_id"].map(dict(enumerate(CLASS_NAMES)))
SMALL = 32 * 32 / (720 * 540)
stats = boxes.groupby("class").agg(
    boxes=("area_frac", "size"),
    median_area_pct=("area_frac", lambda a: 100 * a.median()),
    q10_area_pct=("area_frac", lambda a: 100 * a.quantile(0.1)),
    q90_area_pct=("area_frac", lambda a: 100 * a.quantile(0.9)),
    small_pct=("area_frac", lambda a: 100 * (a < SMALL).mean()),
    median_aspect_wh=("aspect", "median"),
).round(2)
display(stats)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
data = [boxes.loc[boxes["class"] == c, "area_frac"] * 100 for c in CLASS_NAMES]
axes[0].boxplot(data, vert=False); axes[0].set_yticks(range(1, len(CLASS_NAMES) + 1), CLASS_NAMES); axes[0].set_xscale("log")
axes[0].axvline(SMALL * 100, ls="--", c="grey"); axes[0].set_xlabel("box area (% of image, log scale)")
axes[0].set_title("Defect size by class (dashed: COCO 'small')")
for k, c in enumerate(CLASS_NAMES):
    sub = boxes[boxes["class"] == c]
    axes[1].scatter(sub["cx"], sub["cy"], s=4, alpha=0.4, label=c)
axes[1].invert_yaxis(); axes[1].set_xlim(0, 1); axes[1].set_xlabel("x (normalised)"); axes[1].set_ylabel("y")
axes[1].set_title("Box centres (image coordinates)"); axes[1].legend(markerscale=4, fontsize=8)
fig.tight_layout(); fig.savefig(FIG / "01_box_statistics.png", dpi=110); plt.show()

## 4. Defects per image and class co-occurrence

Counted on **unique originals** of the training split (augmented copies would triple-count).

In [ ]:
from porkqc.data.audit import image_classes, original_stem
rows = []
for s in SPLITS:
    for f, cls in image_classes(data_root / s, len(CLASS_NAMES)).items():
        rows.append({"split": s, "file": f, "original": original_stem(f), "classes": cls})
imgs = pd.DataFrame(rows)
train_orig = imgs[imgs.split == "train"].drop_duplicates("original")

co = np.zeros((len(CLASS_NAMES),) * 2, dtype=int)
for cls in train_orig["classes"]:
    for a in cls:
        for b in cls:
            co[a, b] += 1
co_df = pd.DataFrame(co, index=CLASS_NAMES, columns=CLASS_NAMES)
print("Images (train originals) containing class i (row) AND class j (column); diagonal = images with class i")
display(co_df)

n_cls = train_orig["classes"].map(len)
print("Distinct defect classes per training original:", n_cls.value_counts().sort_index().to_dict())
bpi = boxes[boxes.split == "train"].drop_duplicates(["original", "cx", "cy", "class_id"]).groupby("original").size()
print("Boxes per training original: median", bpi.median(), "| max", bpi.max())

## 5. Augmentation accounting and direct leakage

Roboflow keeps the original photo's name before `.rf.` in every exported copy, so we can
count copies per original and check whether any original appears in two splits.

In [ ]:
copies = imgs[imgs.split == "train"].groupby("original").size()
print("Training copies per original:", copies.value_counts().sort_index().to_dict())
by_split = {s: set(imgs.loc[imgs.split == s, "original"]) for s in SPLITS}
shared = {f"{a}&{b}": sorted(by_split[a] & by_split[b]) for a, b in [("train", "valid"), ("train", "test"), ("valid", "test")]}
for k, v in shared.items():
    print(f"originals shared by {k}: {len(v)}", v[:5])
print("Example original names:", sorted(by_split["train"])[:8])

## 6. Near-duplicate leakage (perceptual hashing)

Every image gets a 64-bit difference hash (`porkqc.data.duplicates`). The Hamming distance
between hashes (0–64 bits) measures visual similarity. Instead of guessing a threshold we
calibrate it:

* **Positive control** — pairs of augmented copies of the *same* original (only colour
  changed). These are duplicates by construction; the threshold must accept them.
* **Negative control** — random pairs of *different* originals. Most should be far apart.

Then: for each VALID/TEST image, how close is the nearest TRAIN image?

In [ ]:
import itertools
from porkqc.data.audit import list_images
from porkqc.data.duplicates import hash_images, paired_hamming, nearest_neighbours

files, split_of, hashes = [], [], []
for s in SPLITS:
    paths = list_images(data_root / s / "images")
    files += [p.name for p in paths]; split_of += [s] * len(paths)
    hashes.append(hash_images(paths))
H = np.vstack(hashes); files = np.array(files); split_of = np.array(split_of)
stems = np.array([original_stem(f) for f in files])
print("hashed", len(files), "images")

tr = np.where(split_of == "train")[0]
pairs = [(i, j) for _, idx in pd.Series(tr).groupby(stems[tr]) for i, j in itertools.combinations(idx.values, 2)]
pi, pj = (np.array(x, dtype=int) for x in zip(*pairs)) if pairs else (np.array([], int), np.array([], int))
pos = paired_hamming(H[pi], H[pj]) if pairs else np.array([])
rng = np.random.default_rng(0)
a, b = rng.choice(tr, 20000), rng.choice(tr, 20000)
keep = stems[a] != stems[b]
neg = paired_hamming(H[a[keep]], H[b[keep]])

THRESH = int(np.percentile(pos, 99)) if len(pos) else 6
print(f"positive control (same original): n={len(pos)}, median={np.median(pos):.1f}, 99th pct={THRESH}")
print(f"negative control (different originals): n={len(neg)}, median={np.median(neg):.1f}, "
      f"share <= threshold: {100 * (neg <= THRESH).mean():.2f}%")

nn = {}
for s in ("valid", "test"):
    q = np.where(split_of == s)[0]
    d, j = nearest_neighbours(H[q], H[tr])
    nn[s] = pd.DataFrame({"file": files[q], "nn_train_file": files[tr][j], "distance": d})
    print(f"{s}: {100 * (d <= THRESH).mean():.1f}% of images have a TRAIN image within {THRESH} bits "
          f"(median nearest distance {np.median(d):.0f})")

# Sensitivity to the threshold: colour-only augmentations may be stricter than real
# neighbouring video frames (which also shift), so report several thresholds.
sens = pd.DataFrame([{
    "threshold (bits)": t,
    "valid with train neighbour %": round(100 * (nn["valid"].distance <= t).mean(), 1),
    "test with train neighbour %": round(100 * (nn["test"].distance <= t).mean(), 1),
    "different-original pairs flagged % (false alarms)": round(100 * (neg <= t).mean(), 2),
} for t in sorted({THRESH, 4, 6, 8, 10, 12})])
display(sens)

fig, ax = plt.subplots(figsize=(9, 3.8))
bins = np.arange(0, 65)
ax.hist(pos, bins=bins, alpha=0.6, density=True, label="same original (augmented copies)")
ax.hist(neg, bins=bins, alpha=0.6, density=True, label="different originals (random pairs)")
for s, c in (("valid", "k"), ("test", "r")):
    ax.hist(nn[s]["distance"], bins=bins, histtype="step", lw=2, density=True, color=c, label=f"{s} -> nearest train")
ax.axvline(THRESH + 0.5, ls="--", c="grey", label=f"threshold {THRESH}")
ax.set_xlabel("Hamming distance between dHashes (bits)"); ax.set_ylabel("density"); ax.legend(fontsize=8)
ax.set_title("Near-duplicate check: VALID/TEST vs TRAIN")
fig.tight_layout(); fig.savefig(FIG / "01_near_duplicates.png", dpi=110); plt.show()

### 6.1 Manual check of the closest pairs

Criterion for a true near-duplicate: **same tray and same meat arrangement** (only lighting,
colour or a small shift differs). Look at the closest pairs and the pairs near the threshold.

In [ ]:
from PIL import Image
both = pd.concat([nn["valid"].assign(split="valid"), nn["test"].assign(split="test")]).sort_values("distance")
show = pd.concat([both.head(6), both[both.distance.between(THRESH, THRESH + 3)].head(3)])
fig, axes = plt.subplots(len(show), 2, figsize=(8, 2.9 * len(show)))
for row, (ax1, ax2) in zip(show.itertuples(), np.atleast_2d(axes)):
    for ax, (sp, f) in ((ax1, (row.split, row.file)), (ax2, ("train", row.nn_train_file))):
        with Image.open(data_root / sp / "images" / f) as im:
            ax.imshow(im.convert("RGB"))
        ax.set_title(f"{sp}: {f[:30]}", fontsize=7); ax.axis("off")
    ax1.text(5, 30, f"distance {row.distance}", color="yellow", fontsize=9, backgroundcolor="black")
fig.tight_layout(); fig.savefig(FIG / "01_closest_pairs.png", dpi=90); plt.show()

## 7. Near-duplicate groups across the whole dataset

Images within the threshold (single linkage) and copies of the same original are merged
into groups. Groups spanning several splits are the leakage paths. The group file is saved
for a grouped bootstrap / grouped re-split if needed (D-008).

In [ ]:
from porkqc.data.duplicates import group_by_threshold
groups = np.array(group_by_threshold(H, THRESH, labels=list(stems)))
gdf = pd.DataFrame({"file": files, "split": split_of, "original": stems, "group": groups})
span = gdf.groupby("group")["split"].nunique()
sizes = gdf.groupby("group").size()
print(f"{gdf.group.nunique()} groups for {len(gdf)} images ({gdf.original.nunique()} originals)")
print("group size (images): median", sizes.median(), "| max", sizes.max())
leaky = span[span > 1].index
for s in ("valid", "test"):
    sub = gdf[gdf.split == s]
    print(f"{s}: {100 * sub.group.isin(leaky).mean():.1f}% of images belong to a group that also has TRAIN/other-split images")
gdf.to_csv(AUD / "near_duplicate_groups.csv", index=False)

## 8. Save the audit

In [ ]:
audit = {
    "git_commit": git_commit,
    "overview": overview.reset_index().to_dict(orient="records"),
    "objects_per_class": per_class.to_dict(),
    "box_stats": stats.reset_index().to_dict(orient="records"),
    "cooccurrence_train_originals": co_df.to_dict(),
    "train_copies_per_original": {int(k): int(v) for k, v in copies.value_counts().items()},
    "shared_originals": {k: len(v) for k, v in shared.items()},
    "hash_threshold_bits": THRESH,
    "positive_control": {"n": int(len(pos)), "median": float(np.median(pos)), "p99": THRESH},
    "negative_control": {"n": int(len(neg)), "median": float(np.median(neg)), "share_le_threshold": float((neg <= THRESH).mean())},
    "threshold_sensitivity": sens.to_dict(orient="records"),
    "nearest_train": {s: {"median": float(nn[s].distance.median()), "share_le_threshold": float((nn[s].distance <= THRESH).mean())} for s in nn},
    "groups": {"n_groups": int(gdf.group.nunique()), "leaky_share": {s: float(gdf[gdf.split == s].group.isin(leaky).mean()) for s in ("valid", "test")}},
}
with open(AUD / "audit_summary.json", "w") as fh:
    json.dump(audit, fh, indent=1, default=str)
print(json.dumps(audit, indent=1, default=str)[:3000])

import shutil
from porkqc.env import get_secret
from porkqc.colab_utils import push_results
paths = [AUD, FIG / "01_box_statistics.png", FIG / "01_near_duplicates.png", FIG / "01_closest_pairs.png"]
try:
    sha = push_results(REPO_DIR, [str(p) for p in paths], "results: data audit (Colab)", get_secret("GITHUB_TOKEN"))
    print("Pushed to 'colab-results':", sha[:7])
except (RuntimeError, ValueError) as exc:
    print("Not pushed:", exc)
    archive = shutil.make_archive("/content/results_audit", "zip", ".", "results")
    from google.colab import files as colab_files
    colab_files.download(archive)

## 9. Hand back
Tell Claude "audit pushed" (or attach the zip), and optionally *File → Save a copy in GitHub*
(path `notebooks/01_data_audit.ipynb`) so the printed outputs are kept.